# CVE/CWE → Attack Family Mapping

Dataset: `CVE_CWE_2025.csv`

## 1. Import Libraries

Start by importing what we need: pandas for data handling, plus the CWE hierarchy file.

In [1]:
import pandas as pd
import numpy as np

from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

In [2]:
cwe_df = pd.read_csv("CVE_CWE_2025.csv")
cwe_df.head()

,ID,CVE-ID,CVSS-V4,CVSS-V3,CVSS-V2,SEVERITY,DESCRIPTION,CWE-ID
0,1,CVE-1999-0001,NaN,NaN,5.0,MEDIUM,ip_input.c in BSD-derived TCP/IP implementatio...,CWE-20
1,2,CVE-1999-0002,NaN,NaN,10.0,HIGH,Buffer overflow in NFS mountd gives root acces...,CWE-119
2,3,CVE-1999-0003,NaN,NaN,10.0,HIGH,Execute commands as root via buffer overflow i...,NVD-CWE-Other
3,4,CVE-1999-0004,NaN,NaN,5.0,MEDIUM,"MIME buffer overflow in email clients, e.g. So...",NVD-CWE-Other
4,5,CVE-1999-0005,NaN,NaN,10.0,HIGH,Arbitrary command execution via IMAP buffer ov...,NVD-CWE-Other


In [3]:
cwe_df["CWE-ID"].value_counts().head(30)

CWE-ID
NVD-CWE-Other    55550
CWE-79           35698
CWE-89           13925
CWE-119          12053
CWE-20           10851
CWE-787           9163
CWE-200           8587
CWE-352           7446
CWE-125           6786
CWE-22            6737
CWE-416           5397
CWE-264           5339
CWE-862           4441
CWE-94            4044
CWE-78            3959
CWE-476           3621
CWE-287           3360
CWE-284           3203
CWE-120           2950
CWE-434           2780
CWE-399           2637
CWE-190           2459
CWE-310           2297
CWE-400           2241
CWE-77            2162
CWE-74            1971
CWE-269           1967
CWE-121           1779
CWE-502           1707
CWE-863           1701
Name: count, dtype: int64

## 2. Build CWE/CVE → Attack Family Mapping

Attack family maps individual CWE-IDs into 13 broader attack families (Injection, XSS, 
Memory Corruption, etc.) based on the MITRE CWE hierarchy and frequency 
analysis of the top CWEs in our dataset.

In [4]:
attack_family = {
    # Injection-related
    "CWE-89": "Injection",        # SQL Injection
    "CWE-78": "Injection",        # OS Command Injection
    "CWE-77": "Injection",        # Command Injection (general)
    "CWE-94": "Injection",        # Code Injection
    "CWE-74": "Injection",        # Injection (general/base)
    "CWE-20": "Injection",        # Improper Input Validation (judgment call)
    "CWE-918": "Injection",       # SSRF
    "CWE-611": "Injection",       # XXE
    "CWE-427": "Injection",       # Uncontrolled Search Path Element
    "CWE-601": "Injection",      
    
    # Cross-Site Scripting
    "CWE-79": "XSS",

    # Memory Corruption
    "CWE-119": "Memory Corruption",  # Buffer overflow (general)
    "CWE-787": "Memory Corruption",  # Out-of-bounds Write
    "CWE-125": "Memory Corruption",  # Out-of-bounds Read
    "CWE-416": "Memory Corruption",  # Use After Free
    "CWE-476": "Memory Corruption",  # NULL Pointer Dereference
    "CWE-120": "Memory Corruption",  # Buffer Copy w/o Checking Size
    "CWE-190": "Memory Corruption",  # Integer Overflow
    "CWE-121": "Memory Corruption",  # Stack-based Buffer Overflow
    "CWE-122": "Memory Corruption",  # Heap-based Buffer Overflow
    "CWE-189": "Memory Corruption",  # Numeric Errors

    # Info Disclosure
    "CWE-200": "Info Disclosure",
    "CWE-532": "Info Disclosure",    # Insertion of Sensitive Info into Log Files

    # CSRF
    "CWE-352": "CSRF",

    # Path Traversal
    "CWE-22": "Path Traversal",
    "CWE-59": "Path Traversal",      # Improper Link Resolution (symlink attacks)

    # Authentication & Access Control
    "CWE-264": "Authentication & Access Control",  # Permissions/Privileges/Access Control
    "CWE-284": "Authentication & Access Control",  # Improper Access Control
    "CWE-862": "Authentication & Access Control",  # Missing Authorization
    "CWE-863": "Authentication & Access Control",  # Incorrect Authorization
    "CWE-269": "Authentication & Access Control",  # Improper Privilege Management
    "CWE-287": "Authentication & Access Control",  # Improper Authentication
    "CWE-306": "Authentication & Access Control",  # Missing Authentication for Critical Function
    "CWE-732": "Authentication & Access Control",  # Incorrect Permission Assignment
    "CWE-798": "Authentication & Access Control",  # Hard-coded Credentials
    "CWE-276": "Authentication & Access Control",  # Incorrect Default Permissions
    "CWE-522": "Authentication & Access Control",  # Insufficiently Protected Credentials
    "CWE-639": "Authentication & Access Control",  # Authorization Bypass via User-Controlled Key
    "CWE-255": "Authentication & Access Control",  # Credentials Management Errors

    # File Handling
    "CWE-434": "File Handling",  # Unrestricted Upload of File with Dangerous Type

    # Denial of Service
    "CWE-400": "Denial of Service",  # Uncontrolled Resource Consumption
    "CWE-399": "Denial of Service",  # Resource Management Errors
    "CWE-770": "Denial of Service",  # Allocation of Resources Without Limits
    "CWE-401": "Denial of Service",  # Missing Release of Memory (memory leak)

    # Cryptographic Issues
    "CWE-310": "Cryptographic Issues",
    "CWE-295": "Cryptographic Issues",  # Improper Certificate Validation

    # Deserialization
    "CWE-502": "Deserialization",

    # Race Conditions
    "CWE-362": "Race Condition",

    # NVD's catch-all bucket
    "NVD-CWE-Other": "Other",
}

## 3. Apply the Mapping

Apply the dictionary to create a new `attack_family` column, with any 
unmapped CWE falling back to "Other".

In [5]:
cwe_df["attack_family"] = cwe_df["CWE-ID"].map(attack_family).fillna("Other")
cwe_df["attack_family"].value_counts()

attack_family
Other                              88620
Memory Corruption                  46781
Injection                          41421
XSS                                35698
Authentication & Access Control    27247
Info Disclosure                     9377
Path Traversal                      7917
CSRF                                7446
Denial of Service                   6785
Cryptographic Issues                3341
File Handling                       2780
Deserialization                     1707
Race Condition                      1574
Name: count, dtype: int64

## 4. Checking some of the Labels

I realized that sometimes CWE labels may reflect the software weakness rather than the description, which can lead to inconsistencies between the description and the ID. So I was hoping the NLP would catch that and correct it.

In [6]:
cwe_df[["DESCRIPTION", "attack_family"]].sample(10)

,DESCRIPTION,attack_family
120591,"In kubelet v1.13.6 and v1.14.2, containers for...",Other
179367,A vulnerability exists where an IBM Robotic Pr...,Other
18777,Cross-site scripting (XSS) vulnerability in Mi...,Other
55654,LemonLDAP::NG before 1.2.3 does not use the si...,Authentication & Access Control
46930,win32k.sys in the kernel-mode drivers in Micro...,Other
186658,CKEditor 5 is a JavaScript rich text editor. A...,XSS
158358,The joomsport_md_load AJAX action of the JoomS...,Deserialization
58984,Stack-based buffer overflow in Media Player Cl...,Memory Corruption
137413,A potential vulnerability exists in AMD Platfo...,Injection
212819,The event analysis component in Zoho ManageEng...,Authentication & Access Control


## 5. Define Features (X) and Target (y)

- **X**: the CVE description text (model input)
- **y**: the attack_family label (what we're predicting)

In [7]:
X = cwe_df["DESCRIPTION"]
y = cwe_df["attack_family"]

## 6. Train/Test Split

Split the data 80/20, stratified by `attack_family` to preserve class 
proportions across both sets since we have a class imbalance.

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

## 7. TF-IDF Vectorization

Convert text into numeric features using TF-IDF with unigrams and bigrams 
(`ngram_range=(1,2)`), So that the model can capture two-word phrases like 
"denial of service" rather than only words that are isolated from each other. 

In [9]:
vectorizer = TfidfVectorizer(max_features=15000, stop_words="english", ngram_range=(1,2))
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

## 8. Train Logistic Regression Model

In [10]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train_tfidf, y_train)

,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is '

## 9. Evaluate on Test Set

In [11]:
y_pred = model.predict(X_test_tfidf)

In [12]:
print(classification_report(y_test, y_pred))

                                 precision    recall  f1-score   support

Authentication & Access Control       0.68      0.61      0.65      5450
                           CSRF       0.93      0.92      0.93      1489
           Cryptographic Issues       0.83      0.64      0.72       668
              Denial of Service       0.65      0.41      0.50      1357
                Deserialization       0.90      0.70      0.78       341
                  File Handling       0.80      0.76      0.78       556
                Info Disclosure       0.68      0.49      0.57      1876
                      Injection       0.80      0.74      0.77      8284
              Memory Corruption       0.86      0.87      0.86      9356
                          Other       0.67      0.76      0.71     17724
                 Path Traversal       0.81      0.72      0.77      1583
                 Race Condition       0.80      0.57      0.66       315
                            XSS       0.91      0.

## 10. Investigate Weakest Class (Denial of Service)

In [13]:
cm = confusion_matrix(y_test, y_pred, labels=model.classes_)
cm_df = pd.DataFrame(cm, index=model.classes_, columns=model.classes_)
cm_df.loc["Denial of Service"].sort_values(ascending=False)

Other                              560
Denial of Service                  553
Memory Corruption                  140
Injection                           89
Info Disclosure                      8
Authentication & Access Control      6
Path Traversal                       1
CSRF                                 0
Cryptographic Issues                 0
Deserialization                      0
File Handling                        0
Race Condition                       0
XSS                                  0
Name: Denial of Service, dtype: int64

# 11. Experimenting: Excluding "other" row

In [14]:
mapped_only = cwe_df[cwe_df["attack_family"] != "Other"]
X_mapped = mapped_only["DESCRIPTION"]
y_mapped = mapped_only["attack_family"]

In [15]:
print(f"Rows before: {len(cwe_df)}")
print(f"Rows after excluding Other: {len(mapped_only)}")

Rows before: 280694
Rows after excluding Other: 192074


In [16]:
X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(
    X_mapped, y_mapped, test_size=0.2, random_state=42, stratify=y_mapped
)

In [17]:
vectorizer_m = TfidfVectorizer(max_features=15000, stop_words="english", ngram_range=(1,2))
X_train_tfidf_m = vectorizer_m.fit_transform(X_train_m)
X_test_tfidf_m = vectorizer_m.transform(X_test_m)

In [18]:
model_m = LogisticRegression(max_iter=1000)

In [19]:
model_m.fit(X_train_tfidf_m, y_train_m)

,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is '

In [20]:
y_pred_m = model_m.predict(X_test_tfidf_m)

In [21]:
print(classification_report(y_test_m, y_pred_m))

                                 precision    recall  f1-score   support

Authentication & Access Control       0.79      0.87      0.83      5450
                           CSRF       0.97      0.94      0.96      1489
           Cryptographic Issues       0.88      0.72      0.79       668
              Denial of Service       0.75      0.63      0.68      1357
                Deserialization       0.94      0.72      0.82       341
                  File Handling       0.85      0.80      0.82       556
                Info Disclosure       0.77      0.70      0.73      1876
                      Injection       0.86      0.86      0.86      8284
              Memory Corruption       0.92      0.95      0.93      9356
                 Path Traversal       0.93      0.85      0.89      1583
                 Race Condition       0.90      0.68      0.78       315
                            XSS       0.98      0.97      0.98      7140

                       accuracy                  

## 12. Experimenting with a 50 CWE-IDs

In [22]:
cwe_df["CWE-ID"].value_counts().head(50)

CWE-ID
NVD-CWE-Other    55550
CWE-79           35698
CWE-89           13925
CWE-119          12053
CWE-20           10851
CWE-787           9163
CWE-200           8587
CWE-352           7446
CWE-125           6786
CWE-22            6737
CWE-416           5397
CWE-264           5339
CWE-862           4441
CWE-94            4044
CWE-78            3959
CWE-476           3621
CWE-287           3360
CWE-284           3203
CWE-120           2950
CWE-434           2780
CWE-399           2637
CWE-190           2459
CWE-310           2297
CWE-400           2241
CWE-77            2162
CWE-74            1971
CWE-269           1967
CWE-121           1779
CWE-502           1707
CWE-863           1701
CWE-362           1574
CWE-918           1463
CWE-122           1372
CWE-306           1269
CWE-732           1209
CWE-189           1201
CWE-59            1180
CWE-798           1180
CWE-276           1177
CWE-601           1112
CWE-611           1088
CWE-295           1044
CWE-401            959
CWE-

In [23]:
attack_family_50 = {
    # Injection-related
    "CWE-89": "Injection",        # SQL Injection
    "CWE-78": "Injection",        # OS Command Injection
    "CWE-77": "Injection",        # Command Injection (general)
    "CWE-94": "Injection",        # Code Injection
    "CWE-74": "Injection",        # Injection (general/base)
    "CWE-20": "Injection",        # Improper Input Validation (judgment call)
    "CWE-918": "Injection",       # SSRF
    "CWE-611": "Injection",       # XXE
    "CWE-427": "Injection",       # Uncontrolled Search Path Element
    "CWE-601": "Injection",      
    
    # Cross-Site Scripting
    "CWE-79": "XSS",

    # Memory Corruption
    "CWE-119": "Memory Corruption",  # Buffer overflow (general)
    "CWE-787": "Memory Corruption",  # Out-of-bounds Write
    "CWE-125": "Memory Corruption",  # Out-of-bounds Read
    "CWE-416": "Memory Corruption",  # Use After Free
    "CWE-476": "Memory Corruption",  # NULL Pointer Dereference
    "CWE-120": "Memory Corruption",  # Buffer Copy w/o Checking Size
    "CWE-190": "Memory Corruption",  # Integer Overflow
    "CWE-121": "Memory Corruption",  # Stack-based Buffer Overflow
    "CWE-122": "Memory Corruption",  # Heap-based Buffer Overflow
    "CWE-189": "Memory Corruption",  # Numeric Errors

    # Info Disclosure
    "CWE-200": "Info Disclosure",
    "CWE-532": "Info Disclosure",    # Insertion of Sensitive Info into Log Files

    # CSRF
    "CWE-352": "CSRF",

    # Path Traversal
    "CWE-22": "Path Traversal",
    "CWE-59": "Path Traversal",      # Improper Link Resolution (symlink attacks)

    # Authentication & Access Control
    "CWE-264": "Authentication & Access Control",  # Permissions/Privileges/Access Control
    "CWE-284": "Authentication & Access Control",  # Improper Access Control
    "CWE-862": "Authentication & Access Control",  # Missing Authorization
    "CWE-863": "Authentication & Access Control",  # Incorrect Authorization
    "CWE-269": "Authentication & Access Control",  # Improper Privilege Management
    "CWE-287": "Authentication & Access Control",  # Improper Authentication
    "CWE-306": "Authentication & Access Control",  # Missing Authentication for Critical Function
    "CWE-732": "Authentication & Access Control",  # Incorrect Permission Assignment
    "CWE-798": "Authentication & Access Control",  # Hard-coded Credentials
    "CWE-276": "Authentication & Access Control",  # Incorrect Default Permissions
    "CWE-522": "Authentication & Access Control",  # Insufficiently Protected Credentials
    "CWE-639": "Authentication & Access Control",  # Authorization Bypass via User-Controlled Key
    "CWE-255": "Authentication & Access Control",  # Credentials Management Errors
    "CWE-285": "Authentication & Access Control",
    # File Handling
    "CWE-434": "File Handling",  # Unrestricted Upload of File with Dangerous Type

    # Denial of Service
    "CWE-400": "Denial of Service",  # Uncontrolled Resource Consumption
    "CWE-399": "Denial of Service",  # Resource Management Errors
    "CWE-770": "Denial of Service",  # Allocation of Resources Without Limits
    "CWE-401": "Denial of Service",  # Missing Release of Memory (memory leak)

    # Cryptographic Issues
    "CWE-310": "Cryptographic Issues",
    "CWE-295": "Cryptographic Issues",  # Improper Certificate Validation

    # Deserialization
    "CWE-502": "Deserialization",

    # Race Conditions
    "CWE-362": "Race Condition",

    # NVD's catch-all bucket
    "NVD-CWE-Other": "Other",
}

In [24]:
cwe_df["attack_family_50"] = cwe_df["CWE-ID"].map(attack_family_50).fillna("Other")
cwe_df["attack_family_50"].value_counts()

attack_family_50
Other                              87922
Memory Corruption                  46781
Injection                          41421
XSS                                35698
Authentication & Access Control    27945
Info Disclosure                     9377
Path Traversal                      7917
CSRF                                7446
Denial of Service                   6785
Cryptographic Issues                3341
File Handling                       2780
Deserialization                     1707
Race Condition                      1574
Name: count, dtype: int64

In [25]:
X_50 = cwe_df["DESCRIPTION"]
y_50 = cwe_df["attack_family_50"]

In [26]:
X_train_50, X_test_50, y_train_50, y_test_50 = train_test_split(X_50, y_50, test_size=0.2, random_state=42, stratify=y_50)

In [27]:
vectorizer_50 = TfidfVectorizer(max_features=15000, stop_words="english", ngram_range=(1,2))
X_train_tfidf_50 = vectorizer_50.fit_transform(X_train_50)
X_test_tfidf_50 = vectorizer_50.transform(X_test_50)

In [28]:
model_50 = LogisticRegression(max_iter=1000)
model_50.fit(X_train_tfidf_50, y_train_50)

,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is '

In [29]:
y_pred_50 = model_50.predict(X_test_tfidf_50)
print(classification_report(y_test_50, y_pred_50))

                                 precision    recall  f1-score   support

Authentication & Access Control       0.68      0.62      0.65      5589
                           CSRF       0.94      0.92      0.93      1489
           Cryptographic Issues       0.81      0.67      0.73       668
              Denial of Service       0.62      0.40      0.48      1357
                Deserialization       0.91      0.67      0.78       341
                  File Handling       0.78      0.74      0.76       556
                Info Disclosure       0.64      0.48      0.55      1876
                      Injection       0.80      0.74      0.77      8284
              Memory Corruption       0.86      0.87      0.86      9356
                          Other       0.67      0.75      0.71     17585
                 Path Traversal       0.81      0.73      0.77      1583
                 Race Condition       0.77      0.52      0.62       315
                            XSS       0.91      0.

In [30]:
pd.set_option("display.max_rows", 150)
cwe_df["CWE-ID"].value_counts().iloc[50:150]

CWE-ID
CWE-319     632
CWE-835     601
CWE-312     566
CWE-415     565
CWE-843     547
CWE-203     536
CWE-426     522
CWE-617     498
CWE-327     465
CWE-347     463
CWE-908     454
CWE-129     440
CWE-404     428
CWE-668     417
CWE-311     413
CWE-209     412
CWE-367     399
CWE-755     386
CWE-772     384
CWE-307     381
CWE-254     381
CWE-667     379
CWE-754     376
CWE-290     375
CWE-345     375
CWE-266     368
CWE-369     349
CWE-613     345
CWE-326     344
CWE-1321    340
CWE-134     335
CWE-126     319
CWE-384     318
CWE-16      316
CWE-346     312
CWE-552     301
CWE-665     297
CWE-1021    297
CWE-191     283
CWE-80      281
CWE-281     279
CWE-288     278
CWE-98      260
CWE-922     259
CWE-1333    258
CWE-23      258
CWE-693     257
CWE-330     252
CWE-674     242
CWE-444     232
CWE-19      230
CWE-1236    230
CWE-824     223
CWE-73      205
CWE-88      204
CWE-116     198
CWE-707     197
CWE-428     192
CWE-521     188
CWE-250     185
CWE-640     175
CWE-17      166
C

## 13. Testing with 150 CWE-IDs

In [31]:
attack_family_150_additions = {
    "CWE-319": "Cryptographic Issues", 
    "CWE-312": "Cryptographic Issues",
    "CWE-327": "Cryptographic Issues", 
    "CWE-347": "Cryptographic Issues",
    "CWE-311": "Cryptographic Issues", 
    "CWE-326": "Cryptographic Issues",
    "CWE-345": "Cryptographic Issues", 
    "CWE-330": "Cryptographic Issues",
    "CWE-321": "Cryptographic Issues", 
    "CWE-338": "Cryptographic Issues",

    "CWE-415": "Memory Corruption", 
    "CWE-843": "Memory Corruption",
    "CWE-908": "Memory Corruption", 
    "CWE-129": "Memory Corruption",
    "CWE-126": "Memory Corruption", 
    "CWE-665": "Memory Corruption",
    "CWE-191": "Memory Corruption", 
    "CWE-369": "Memory Corruption",
    "CWE-193": "Memory Corruption", 
    "CWE-824": "Memory Corruption",
    "CWE-788": "Memory Corruption", 
    "CWE-131": "Memory Corruption",
    "CWE-680": "Memory Corruption",

    "CWE-835": "Denial of Service", 
    "CWE-617": "Denial of Service",
    "CWE-404": "Denial of Service", 
    "CWE-755": "Denial of Service",
    "CWE-772": "Denial of Service", 
    "CWE-674": "Denial of Service",
    "CWE-754": "Denial of Service", 
    "CWE-459": "Denial of Service",
    "CWE-248": "Denial of Service", 
    "CWE-703": "Denial of Service",

    "CWE-203": "Info Disclosure", 
    "CWE-209": "Info Disclosure",
    "CWE-668": "Info Disclosure", 
    "CWE-552": "Info Disclosure",
    "CWE-922": "Info Disclosure", 
    "CWE-201": "Info Disclosure",
    "CWE-359": "Info Disclosure", 
    "CWE-497": "Info Disclosure",

    "CWE-290": "Authentication & Access Control", 
    "CWE-613": "Authentication & Access Control",
    "CWE-266": "Authentication & Access Control", 
    "CWE-281": "Authentication & Access Control",
    "CWE-288": "Authentication & Access Control", 
    "CWE-346": "Authentication & Access Control",
    "CWE-693": "Authentication & Access Control", 
    "CWE-307": "Authentication & Access Control",
    "CWE-384": "Authentication & Access Control", 
    "CWE-521": "Authentication & Access Control",
    "CWE-294": "Authentication & Access Control", 
    "CWE-259": "Authentication & Access Control",
    "CWE-425": "Authentication & Access Control",
    "CWE-1021": "Authentication & Access Control",
    
    "CWE-23": "Path Traversal", 
    "CWE-98": "Path Traversal",

    "CWE-80": "XSS", 

    "CWE-134": "Injection", "CWE-426": 
    "Injection", "CWE-444": "Injection",
    "CWE-73": "Injection", "CWE-88": 
    "Injection", "CWE-116": "Injection",
    "CWE-610": "Injection", "CWE-829": 
    "Injection", "CWE-91": "Injection",

    "CWE-367": "Race Condition", 
    "CWE-667": "Race Condition",
}

In [32]:
attack_family_150 = {**attack_family, **attack_family_150_additions}

In [33]:
cwe_df["attack_family_150"] = cwe_df["CWE-ID"].map(attack_family_150).fillna("Other")
cwe_df["attack_family_150"].value_counts()

attack_family_150
Other                              67934
Memory Corruption                  50731
Injection                          43441
XSS                                35979
Authentication & Access Control    31084
Info Disclosure                    11634
Denial of Service                  10052
Path Traversal                      8435
CSRF                                7446
Cryptographic Issues                7119
File Handling                       2780
Race Condition                      2352
Deserialization                     1707
Name: count, dtype: int64

In [34]:
X_150 = cwe_df["DESCRIPTION"]
y_150 = cwe_df["attack_family_150"]

In [35]:
X_train_150, X_test_150, y_train_150, y_test_150 = train_test_split(X_150, y_150, test_size=0.2, random_state=42, stratify=y_150)

In [36]:
vectorizer_150 = TfidfVectorizer(max_features=15000, stop_words="english", ngram_range=(1,2))
X_train_tfidf_150 = vectorizer_150.fit_transform(X_train_150)
X_test_tfidf_150 = vectorizer_150.transform(X_test_150)

In [37]:
model_150 = LogisticRegression(max_iter=1000)
model_150.fit(X_train_tfidf_150, y_train_150)

,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is '

In [38]:
y_pred_150 = model_150.predict(X_test_tfidf_150)
print(classification_report(y_test_150, y_pred_150))

                                 precision    recall  f1-score   support

Authentication & Access Control       0.67      0.70      0.69      6217
                           CSRF       0.94      0.92      0.93      1489
           Cryptographic Issues       0.73      0.61      0.67      1424
              Denial of Service       0.68      0.55      0.61      2011
                Deserialization       0.90      0.69      0.78       341
                  File Handling       0.80      0.74      0.77       556
                Info Disclosure       0.64      0.57      0.60      2327
                      Injection       0.78      0.75      0.76      8688
              Memory Corruption       0.85      0.89      0.87     10146
                          Other       0.63      0.66      0.64     13587
                 Path Traversal       0.80      0.75      0.78      1687
                 Race Condition       0.80      0.56      0.66       470
                            XSS       0.92      0.

In [39]:
mapped_only_150 = cwe_df[cwe_df["attack_family_150"] != "Other"]
X_mapped_150 = mapped_only_150["DESCRIPTION"]
y_mapped_150 = mapped_only_150["attack_family_150"]

print(f"Rows before: {len(cwe_df)}")
print(f"Rows after excluding Other: {len(mapped_only_150)}")

Rows before: 280694
Rows after excluding Other: 212760


In [40]:
X_train_150m, X_test_150m, y_train_150m, y_test_150m = train_test_split(
    X_mapped_150, y_mapped_150, test_size=0.2, random_state=42, stratify=y_mapped_150
)

In [41]:
vectorizer_150m = TfidfVectorizer(max_features=15000, stop_words="english", ngram_range=(1,2))
X_train_tfidf_150m = vectorizer_150m.fit_transform(X_train_150m)
X_test_tfidf_150m = vectorizer_150m.transform(X_test_150m)

In [42]:
model_150m = LogisticRegression(max_iter=1000)
model_150m.fit(X_train_tfidf_150m, y_train_150m)

,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is '

In [43]:
y_pred_150m = model_150m.predict(X_test_tfidf_150m)
print(classification_report(y_test_150m, y_pred_150m))

                                 precision    recall  f1-score   support

Authentication & Access Control       0.76      0.84      0.80      6217
                           CSRF       0.98      0.93      0.95      1489
           Cryptographic Issues       0.80      0.67      0.73      1424
              Denial of Service       0.74      0.68      0.71      2010
                Deserialization       0.93      0.72      0.81       341
                  File Handling       0.84      0.77      0.80       556
                Info Disclosure       0.72      0.69      0.70      2327
                      Injection       0.84      0.84      0.84      8688
              Memory Corruption       0.91      0.94      0.92     10146
                 Path Traversal       0.91      0.84      0.88      1687
                 Race Condition       0.88      0.62      0.73       471
                            XSS       0.98      0.97      0.98      7196

                       accuracy                  

## Summary

I tested a few variations before settling on this model on step 9:
- Baseline (unigrams, no class weighting): 0.74 accuracy
- Balanced class weights: 0.68 accuracy (higher recall on rare classes, 
  but much lower precision)
- Balanced + bigrams: 0.71 accuracy
- Bigrams only (final, above): 0.77 accuracy
- 12-class (Other excluded, step 11): 0.89 accuracy, 0.84 macro F1
- No big changes when it comes to adding up to 50 CWE-IDs, Actually went down when it increased to 150 CWE-IDs